<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/homework_solutions/cem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross entropy method

In [ ]:
from typing import Callable, List

import jax
import jax.numpy as jnp
import matplotlib.animation as animation
import matplotlib.pyplot as plt
from ipywidgets import interact


In [ ]:
# helper function to make a video of the CEM progress
def make_cem_video_and_gif(
    plotting_func: Callable[[bool], None],
    lower_limit: jnp.ndarray,
    upper_limit: jnp.ndarray,
    sample_list: List[jnp.ndarray],
    filename_gif: str = "cem_progress.gif",
):
    """Make a video of the CEM progress.

    Args:
        plotting_func: A function that plots the function landscape progress.
        lower_limit: The lower limit of the search space.
        upper_limit: The upper limit of the search space.
        sample_list: A list of samples.
        filename_gif: The filename of the GIF.
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    plotting_func(colorbar=True)

    def update(i):
        ax.clear()
        ax.set_title(f"CEM Iteration {i}")
        plotting_func(colorbar=False)
        ax.scatter(
            sample_list[i][:, 0], sample_list[i][:, 1], alpha=0.3, color="yellow"
        )
        ax.set_xlim(lower_limit[0], upper_limit[0])
        ax.set_ylim(lower_limit[1], upper_limit[1])
        return ax

    ani = animation.FuncAnimation(
        fig, update, frames=len(sample_list), blit=False, repeat=False
    )

    # Save as GIF -- requires imagemagick or pillow installed
    ani.save(filename_gif, writer="pillow", fps=5)

    plt.close(fig)
    print(f"Saved CEM progress gif as {filename_gif}")

In [ ]:
@jax.jit
def branin_func(
    x, y, a=1, b=5.1 / (4 * jnp.pi**2), c=5 / jnp.pi, r=6, s=10, t=1 / (8 * jnp.pi)
):
    return a * (y - b * x**2 + c * x - r) ** 2 + s * (1 - t) * jnp.cos(x) + s


def plot_branin(colorbar=True):
    N = 51
    X, Y = jnp.meshgrid(jnp.linspace(-5, 10, N), jnp.linspace(0, 15, N))
    Z = branin_func(X, Y)
    plt.contourf(X, Y, Z, levels=20, cmap="viridis")
    if colorbar:
        plt.colorbar()
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("Branin Function")

In [ ]:
def cross_entropy_method(
    cost_func,
    n_iterations=10,
    n_samples=512,
    m_elite=64,
    seed=1,
    lower_limit=jnp.array([-5.0, 0.0]),
    upper_limit=jnp.array([10.0, 15.0]),
):

    def mean_and_cov(X):
        """
        X: array-like of shape (n_samples, n_features)
        sample=True => sample covariance (n-1); False => population (n)
        Returns: (mean vector, covariance matrix)
        """
        # X = np.asarray(X, dtype=float)
        if len(X.shape) == 1:
            X = X[:, None]
        n = X.shape[0]
        if n < 2:
            raise ValueError("Need at least 2 samples")

        mean = X.mean(axis=0)
        Xc = X - mean
        cov = (Xc.T @ Xc) / n
        return mean, cov

    samples = jax.random.uniform(
        jax.random.PRNGKey(seed), (n_samples, 2), minval=lower_limit, maxval=upper_limit
    )  # initialize samples uniformly in the given limits

    sample_list = [samples]
    for _ in range(n_iterations):
        values = jax.vmap(lambda s: cost_func(s[0], s[1]))(
            samples
        )  # evaluate the function on the samples
        ordered_indices = jnp.argsort(
            values
        )  # get the indices that would sort the values
        ordered_samples = samples[
            ordered_indices
        ]  # sort the samples according to their function values
        elite_samples = ordered_samples[:m_elite]  # select the top m_elite samples
        sample_list.append(elite_samples)  # store the elite samples for this iteration
        mean, cov = mean_and_cov(
            elite_samples
        )  # compute the mean and covariance of the elite samples
        samples = jax.random.multivariate_normal(
            jax.random.PRNGKey(1), mean, cov, (n_samples,)
        )  # sample new candidates from the distribution defined by the mean and covariance of the elite samples
    return mean, cov, sample_list

Set up CEM problem for Branin function

In [ ]:
lower_limit = jnp.array([-5.0, 0.0])
upper_limit = jnp.array([10.0, 15.0])
seed = 3  # try different seeds
n_samples = 1024  # try different number of samples
m_elite = int(n_samples * 0.5)  # try different number of elite samples
n_iterations = 15  # try different number of iterations

mean, cov, sample_list = cross_entropy_method(
    branin_func,
    n_iterations=n_iterations,
    n_samples=n_samples,
    m_elite=m_elite,
    seed=seed,
    lower_limit=lower_limit,
    upper_limit=upper_limit,
)


@interact(i=(0, n_iterations - 1))
def plot_cem(i):
    plt.figure(figsize=(8, 6))
    plt.title(f"CEM Iteration {i}")
    plot_branin()
    plt.scatter(sample_list[i][:, 0], sample_list[i][:, 1], alpha=0.3, color="yellow")
    plt.xlim(lower_limit[0], upper_limit[0])
    plt.ylim(lower_limit[1], upper_limit[1])

interactive(children=(IntSlider(value=7, description='i', max=14), Output()), _dom_classes=('widget-interact',…

In [ ]:
make_cem_video_and_gif(
    plot_branin,
    lower_limit,
    upper_limit,
    sample_list,
    "outputs/branin_cem_progress.gif",
)

Saved CEM progress gif as outputs/branin_cem_progress.gif


Implemet CEM to Rosenbrock's function which features the long narrow gobal minimum at (1,1)

In [ ]:
def rosenbrock_func(x, y, a=1, b=2):
    return (a - x) ** 2 + b * (y - x**2) ** 2


def plot_rosenbrock(colorbar=True):
    N = 51
    X, Y = jnp.meshgrid(jnp.linspace(-2, 2, N), jnp.linspace(-2, 2, N))
    Z = rosenbrock_func(X, Y)
    plt.contourf(X, Y, Z, levels=20, cmap="viridis")
    if colorbar:
        plt.colorbar()
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("Rosenbrock Function")


In [ ]:
lower_limit = jnp.array([-2.0, -2.0])
upper_limit = jnp.array([2.0, 2.0])
seed = 3  # try different seeds
n_samples = 1024 * 4  # try different number of samples
m_elite = int(n_samples * 0.5)  # try different number of elite samples
n_iterations = 20  # try different number of iterations
mean_b, cov_b, sample_list = cross_entropy_method(
    rosenbrock_func,
    n_iterations=n_iterations,
    n_samples=n_samples,
    m_elite=m_elite,
    seed=seed,
    lower_limit=lower_limit,
    upper_limit=upper_limit,
)


@interact(i=(0, n_iterations - 1))
def plot_cem(i):
    plt.figure(figsize=(8, 6))
    plt.title(f"CEM Iteration {i}")
    plot_rosenbrock()
    plt.scatter(sample_list[i][:, 0], sample_list[i][:, 1], alpha=0.3, color="yellow")
    plt.xlim(lower_limit[0], upper_limit[0])
    plt.ylim(lower_limit[1], upper_limit[1])
    plt.grid(alpha=0.2)


###### end of solutions

interactive(children=(IntSlider(value=9, description='i', max=19), Output()), _dom_classes=('widget-interact',…

In [ ]:
make_cem_video_and_gif(
    plot_rosenbrock,
    lower_limit,
    upper_limit,
    sample_list,
    "outputs/rosenbrock_cem_progress.gif",
)

Saved CEM progress gif as outputs/rosenbrock_cem_progress.gif
